# Notebook 4: Literature Review — Peer-Reviewed Statistics
Pruned from notebook 06. Removed: club vs school cost comparison (→ 01_costs),
college/pro funnel (→ 02_fools_gold), internationalization (marginal).

Kept: socioeconomic stratification, pro attainment + career survival, relative age effect,
domestic spot squeeze, transfer portal risk, reference table.

Outputs: `income_stratification.png`, `pro_attainment_and_career.png`,
`relative_age_effect.png`, `domestic_spots_squeeze.png`, `portal_risk_and_cascade.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

stats = pd.read_csv('../data/processed/research_statistics.csv')
refs  = pd.read_csv('../data/processed/references.csv')
print(f'{len(stats)} peer-reviewed statistics across {stats["ref_id"].nunique()} sources')
print(refs[['ref_id','authors','year','journal']].to_string(index=False))

## 1. Socioeconomic Stratification (Bocarro et al. 2018 — R002; Bairner et al. 2016 — R001)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income distribution of club sport parents (Bocarro 2018, n=949)
income_buckets = ['< $50k', '$50k–$100k', '> $100k']
income_pct     = [8, 32, 60]
colors = ['#e74c3c', '#f39c12', '#27ae60']
axes[0].bar(income_buckets, income_pct, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('Household Income of Club Soccer Parents\n(Bocarro et al., 2018; n=949)', fontsize=12)
axes[0].set_ylabel('% of Parents')
for i, v in enumerate(income_pct):
    axes[0].text(i, v+1, f'{v}%', ha='center', fontweight='bold')
axes[0].set_ylim(0, 75)

# National youth soccer income (Bairner 2016)
nat_buckets = ['< $25k', '$25k–$75k', '> $75k']
nat_pct     = [11, 56, 33]
axes[1].bar(nat_buckets, nat_pct, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Household Income: National Youth Soccer Participants\n(Bairner et al., 2016)', fontsize=12)
axes[1].set_ylabel('% of Participants')
for i, v in enumerate(nat_pct):
    axes[1].text(i, v+1, f'{v}%', ha='center', fontweight='bold')
axes[1].set_ylim(0, 75)

plt.tight_layout()
plt.savefig('../data/analysis/income_stratification.png', dpi=150)
plt.show()

print('Key finding (Bocarro 2018): 60% of club soccer families earn >$100k')
print('National participation (Bairner 2016): 56% earn $25k–$75k — more representative')
print('The gap grows as competitive tier increases.')

## 2. Professional Attainment Odds & MLS Career Survival
(Beron et al. 2019 — R005; Watanabe et al. 2010 — R006)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Academy → Pro attainment (Beron 2019, n=1,071)
pro_counts = [40, 1031]
pro_labels = [f'Reached Pro First Team\n(n=40, 4.7%)', f'Did Not Reach Pro\n(n=1031, 95.3%)']
axes[0].pie(pro_counts, labels=pro_labels, colors=['#27ae60', '#bdc3c7'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Academy Players Who Reach\nProfessional First Team\n(Beron et al., 2019; n=1,071)', fontsize=12)

# MLS career survival curve (Watanabe 2010)
# ~30% one-and-done; ~12% annual exit rate thereafter
years = np.arange(0, 12)
survival = np.zeros(len(years))
survival[0] = 1.0
survival[1] = 0.70
for i in range(2, len(years)):
    survival[i] = survival[i-1] * (1 - 0.12)

axes[1].plot(years, survival * 100, 'o-', color='#2980b9', linewidth=2.5, markersize=6)
axes[1].axvline(2.4, color='red', linestyle='--', alpha=0.7,
                label='Mean initial career expectancy: 2.4 yrs')
axes[1].axvline(6.6, color='orange', linestyle='--', alpha=0.7,
                label='Expected career if reach yr 4: 6.6 yrs')
axes[1].fill_between(years, survival * 100, alpha=0.15, color='#2980b9')
axes[1].set_xlabel('Career Year')
axes[1].set_ylabel('% of Players Still Active')
axes[1].set_title('MLS Career Survival Curve\n(Watanabe et al., 2010; n=~1,100)', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.savefig('../data/analysis/pro_attainment_and_career.png', dpi=150)
plt.show()

## 3. Relative Age Effect (Beron et al. 2019 — R005)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

quarters    = ['Q1 (Jan–Mar)', 'Q2 (Apr–Jun)', 'Q3 (Jul–Sep)', 'Q4 (Oct–Dec)']
academy_rep = [34, 28, 22, 16]  # actual distribution in youth academies
expected    = [25, 25, 25, 25]  # equal distribution

x = np.arange(len(quarters))
axes[0].bar(x - 0.2, expected,    0.4, label='Expected (equal)', color='#bdc3c7', alpha=0.8)
axes[0].bar(x + 0.2, academy_rep, 0.4, label='Actual in academy', color='#e67e22', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(quarters)
axes[0].set_title('Relative Age Effect in Youth Academies\n(Beron et al., 2019; n=1,071)', fontsize=12)
axes[0].set_ylabel('% of Academy Players')
axes[0].legend()

# Q4 persisters have 3x better pro odds
groups   = ['Q1–Q3 players\n(typical)', 'Q4 players\n(Oct–Dec, who persist)']
pro_odds = [1.0, 3.0]
bar_c    = ['#3498db', '#27ae60']
axes[1].bar(groups, pro_odds, color=bar_c, alpha=0.85, edgecolor='white', width=0.4)
axes[1].axhline(1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Relative Odds of Reaching Pro\nOnce in Academy (Beron et al., 2019)', fontsize=12)
axes[1].set_ylabel('Relative Odds (1.0 = baseline)')
for i, v in enumerate(pro_odds):
    axes[1].text(i, v + 0.05, f'{v}x', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 4)

plt.tight_layout()
plt.savefig('../data/analysis/relative_age_effect.png', dpi=150)
plt.show()

print('RAE insight: Early-born (Q1) players dominate academy selection (34% vs 25% expected)')
print('But Q4 players who overcome selection bias have 3x better pro odds — selection is lossy.')

## 4. Domestic Roster Spot Squeeze (Dure et al. 2018 — R004)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# D1 roster composition pie (28-player cap)
roster_breakdown = {
    'International\n(~33%)': 9.2,
    'Portal transfers\n(~25%)': 7.0,
    'Available for\nHS recruits': 11.8,
}
colors_r = ['#e74c3c', '#e67e22', '#27ae60']
axes[0].pie(roster_breakdown.values(), labels=roster_breakdown.keys(),
            colors=colors_r, autopct='%1.0f players', startangle=90,
            textprops={'fontsize': 10})
axes[0].set_title('Avg D1 Men Roster Composition\n(28-player cap, 2024)', fontsize=12)

# HS recruits per class across scenarios
portal_pct_range = [0.20, 0.25, 0.30]
intl_pct_range   = [0.25, 0.33, 0.40]
scenario_labels  = ['Optimistic\n(low intl, low portal)', 'Mid estimate', 'Pessimistic\n(high intl, high portal)']
hs_per_class = []
for ip, pp in zip(intl_pct_range, portal_pct_range):
    domestic_total = 28 * (1 - ip)
    after_portal   = domestic_total * (1 - pp)
    hs_per_class.append(after_portal / 4)

bar_colors_b = ['#27ae60', '#f39c12', '#e74c3c']
bars = axes[1].bar(scenario_labels, hs_per_class, color=bar_colors_b, alpha=0.85, edgecolor='white')
axes[1].axhline(4.5, color='gray', linestyle='--', alpha=0.6, label='Rule of thumb: 4–5 HS recruits/class')
axes[1].set_title('Est. US High School Players Recruited\nper Class per D1 Program (2024)', fontsize=12)
axes[1].set_ylabel('Players per graduating class')
for bar, val in zip(bars, hs_per_class):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.1f}', ha='center', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 8)

plt.tight_layout()
plt.savefig('../data/analysis/domestic_spots_squeeze.png', dpi=150)
plt.show()

total_d1_hs_low  = int(min(hs_per_class) * 210)
total_d1_hs_high = int(max(hs_per_class) * 210)
print(f'Across ~210 D1 men programs: ~{total_d1_hs_low}–{total_d1_hs_high} US HS slots per year')
print(f'vs ~4,000,000 total youth soccer players = {total_d1_hs_low/4e6*100:.3f}%–{total_d1_hs_high/4e6*100:.3f}%')

## 5. Transfer Portal Risk & Cascade Displacement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Portal outcome
outcomes = ['Find new program\nwith scholarship', 'Portal purgatory\n(no new home)']
pcts     = [50, 50]
colors_p = ['#27ae60', '#e74c3c']
axes[0].pie(pcts, labels=outcomes, colors=colors_p, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title('NCAA Transfer Portal Outcomes\n(~50% of entrants do not find a new program)', fontsize=12)
axes[0].text(0, -1.55,
    'Scholarship is NOT guaranteed once a player enters the portal.',
    ha='center', fontsize=9, color='#c0392b',
    bbox=dict(boxstyle='round', facecolor='#fdecea', alpha=0.8))

# Cascade displacement
levels    = ['Elite D1\n(ACC/Big Ten)', 'Mid-Major D1', 'NCAA D2', 'D3 / NAIA', 'No program\n(pushed out)']
displaced = [150, 300, 400, 350, 200]
colors_c  = ['#9b59b6', '#3498db', '#27ae60', '#f39c12', '#e74c3c']
bars = axes[1].barh(levels[::-1], displaced[::-1], color=colors_c[::-1], alpha=0.85)
axes[1].set_xlabel('Estimated Players Displaced Annually (cascade estimate)')
axes[1].set_title('28-Player Cap Cascade: Players Pushed\nDown or Out per Year (estimate)', fontsize=12)
for bar, val in zip(bars, displaced[::-1]):
    axes[1].text(val + 4, bar.get_y() + bar.get_height()/2, f'~{val}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/analysis/portal_risk_and_cascade.png', dpi=150)
plt.show()

print('Cascade: Elite D1 cuts → portal → displace mid-major → displace D2 → displace D3/NAIA')
print('High school recruits crowded out at ALL division levels simultaneously.')

## 6. Reference Summary Table

In [ ]:
refs_display = refs[['ref_id','authors','year','journal','doi']].copy()
refs_display.columns = ['ID','Authors','Year','Journal','DOI']
refs_display['DOI'] = refs_display['DOI'].apply(lambda d: f'https://doi.org/{d}')
print(refs_display.to_string(index=False))